In [1]:
import numpy as np

# Define dimensions
num_channels = 4  # Channels: 1, 2, 3, 4 (indices 0-3)
value_range = (0, 10)  # Values: 0 to 10 (11 values, indices 0-10)
z_range = (0, 48)  # z: 0 to 35 (36 values, indices 0-35)
y_range = (0, 343)  # y: 0 to 343 (344 values, indices 0-343)
x_range = (0, 680)  # x: 0 to 680 (681 values, indices 0-680)

# Calculate array dimensions
num_values = value_range[1] - value_range[0] + 1  # 11 values (0-10)
z_dim = z_range[1] - z_range[0] + 1  # 36 values (0-35)
y_dim = y_range[1] - y_range[0] + 1  # 344 values (0-343)
x_dim = x_range[1] - x_range[0] + 1  # 681 values (0-680)

# Calculate scaling factors
x_scale = x_dim / 172  # Old x_dim was 172
y_scale = y_dim / 87   # Old y_dim was 87

# Create 5D array initialized with zeros
# Shape: (channel, value, z, y, x)
data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

# Generate random values for each (z, y, x) position
for i in range(0, 3):
    for z in range(z_dim):
        # Channel 0: x ranges scaled from original 38+20*i to 43+20*i
        for y in range(y_dim):
            x_start = max(0, int((38+20*i) * x_scale))
            x_end = min(x_dim, int((43+20*i) * x_scale))
            for x in range(x_start, x_end):
                random_value = np.random.randint(6, 11)
                data[0, random_value, z, y, x] = 1
        # Channel 1: y ranges scaled from original 18+20*i to 23+20*i
        y_start = max(0, int((18+20*i) * y_scale))
        y_end = min(y_dim, int((23+20*i) * y_scale))
        for y in range(y_start, y_end):
            for x in range(x_dim):
                random_value = np.random.randint(6, 11)
                data[1, random_value, z, y, x] = 1
        # Channel 2: diagonal lines with slope 1 (y = x + c)
        strip_width = int(3 * max(x_scale, y_scale))  # Scaled strip width
        c_values = [int(-60 * x_scale), 0]  # Scaled c values
        for c in c_values:
            for y in range(y_dim):
                for x in range(x_dim):
                    if abs(y - x - c) <= strip_width:
                        random_value = np.random.randint(6, 11)
                        data[2, random_value, z, y, x] = 1
        # Channel 3: diagonal lines with slope -1 (y = -x + d)
        strip_width = int(3 * max(x_scale, y_scale))  # Scaled strip width
        d_values = [int(60 * y_scale), int(120 * y_scale)]  # Scaled d values
        for d in d_values:
            for y in range(y_dim):
                for x in range(x_dim):
                    if abs(y + x - d) <= strip_width:
                        random_value = np.random.randint(6, 11)
                        data[3, random_value, z, y, x] = 1

print(f"Created 5D data array with shape: {data.shape}")
print(f"Dimensions breakdown:")
print(f"  - Channels: {num_channels} (1, 2, 3, 4)")
print(f"  - Values: {num_values} (0 to {value_range[1]})")
print(f"  - Z: {z_dim} (0 to {z_range[1]})")
print(f"  - Y: {y_dim} (0 to {y_range[1]})")
print(f"  - X: {x_dim} (0 to {x_range[1]})")
print(f"\nData type: {data.dtype}")
print(f"Total elements: {data.size:,}")
print(f"Memory size: {data.nbytes / 1024 / 1024:.2f} MB")

# Save the data
np.save('groundtruth.npy', data)
print(f"\n✓ 5D data saved to: groundtruth.npy")


Created 5D data array with shape: (4, 11, 49, 344, 681)
Dimensions breakdown:
  - Channels: 4 (1, 2, 3, 4)
  - Values: 11 (0 to 10)
  - Z: 49 (0 to 48)
  - Y: 344 (0 to 343)
  - X: 681 (0 to 680)

Data type: int8
Total elements: 505,073,184
Memory size: 481.68 MB

✓ 5D data saved to: linear.npy


In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("groundtruth.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0
    "#1f78b4",    # channel 1
    "#b2df8a",     # channel 2
    "#33a02c",  # channel 3
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

for ch in range(n_channels):
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.1
max_points = 5000

for ch in range(n_channels):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,
            color=channel_colors[ch],
            opacity=0.8  # must be single number!
        ),
        name=f"Biomarker {ch}"
    ))

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Interactive View of Channels (Non-Isotropic: Z << X, Y)",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
    ),
    autosize=True,
    showlegend=True,
)

print(f"3D Visualization with real ranges (Non-Isotropic):")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


3D Visualization with real ranges (Non-Isotropic):
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y


In [22]:
import numpy as np

# ============================================================================
# STEP 1: Load the groundtruth data
# ============================================================================
data = np.load("groundtruth.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded data shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3)")
print(f"  Values: {n_values} (0-10)")
print(f"  Spatial dimensions: Z={n_z}, Y={y_dim}, X={x_dim}")

# ============================================================================
# STEP 2: Calculate intensity per voxel
# Sum all channel values and value indices for each voxel position
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 2: Calculating voxel intensity")
print(f"{'='*60}")

# Sum over channels (axis=0) and values (axis=1) to get total intensity per voxel
# Input shape: (n_channels, n_values, n_z, y_dim, x_dim)
# Output shape: (n_z, y_dim, x_dim)
voxel_intensity = np.sum(data, axis=(0, 1))

print(f"Voxel intensity shape: {voxel_intensity.shape}")
print(f"  Range: [{voxel_intensity.min()}, {voxel_intensity.max()}]")
print(f"  Non-zero voxels: {np.sum(voxel_intensity > 0):,}")

# ============================================================================
# STEP 3: Calculate 9x9x9 cube sums with step=9 (no overlap)
# Each cube represents a 9x9x9 region, and cubes don't overlap
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 3: Computing 9x9x9 cube sums (non-overlapping)")
print(f"{'='*60}")

cube_size = 9
step = cube_size  # Step size equals cube size (no overlap)

# Calculate number of cubes in each dimension
num_cubes_z = n_z // cube_size  # e.g., 49 // 9 = 5
num_cubes_y = y_dim // cube_size  # e.g., 344 // 9 = 38
num_cubes_x = x_dim // cube_size  # e.g., 681 // 9 = 75

print(f"Cube size: {cube_size}x{cube_size}x{cube_size}")
print(f"Step size: {step} (no overlap)")
print(f"Number of cubes: Z={num_cubes_z}, Y={num_cubes_y}, X={num_cubes_x}")
print(f"Total possible cubes: {num_cubes_z * num_cubes_y * num_cubes_x:,}")

# Store cube sums and their center positions
cube_data = []  # List of (z_center, y_center, x_center, cube_sum)

# Iterate through non-overlapping cubes
for z_idx in range(num_cubes_z):
    for y_idx in range(num_cubes_y):
        for x_idx in range(num_cubes_x):
            # Calculate cube boundaries
            z_start = z_idx * cube_size
            z_end = z_start + cube_size
            y_start = y_idx * cube_size
            y_end = y_start + cube_size
            x_start = x_idx * cube_size
            x_end = x_start + cube_size
            
            # Sum all values in the 9x9x9 cube
            cube_sum = np.sum(voxel_intensity[z_start:z_end, y_start:y_end, x_start:x_end])
            
            # Calculate center position of the cube
            z_center = z_start + cube_size // 2
            y_center = y_start + cube_size // 2
            x_center = x_start + cube_size // 2
            
            cube_data.append((z_center, y_center, x_center, cube_sum))

print(f"Computed {len(cube_data):,} cube sums")

# ============================================================================
# STEP 4: Find minimum and maximum GT values
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 4: GT value statistics")
print(f"{'='*60}")

cube_sums_list = [cube[3] for cube in cube_data]
gt_min = min(cube_sums_list)
gt_max = max(cube_sums_list)

print(f"  Minimum GT value: {gt_min:.2f}")
print(f"  Maximum GT value: {gt_max:.2f}")
print(f"  Mean GT value: {np.mean(cube_sums_list):.2f}")
print(f"  Non-zero cubes: {sum(1 for s in cube_sums_list if s > 0):,}")

# ============================================================================
# STEP 5: Collect candidates and sort in ascending order
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 5: Collecting and sorting candidates")
print(f"{'='*60}")

# Collect all cubes with cube_sum > 0
candidates = [cube for cube in cube_data if cube[3] > 0]

print(f"Total candidate cubes: {len(candidates):,}")

# Sort candidates by cube sum value in ascending order
candidates.sort(key=lambda x: x[3])
print(f"  Sorted in ascending order (lowest to highest)")

# ============================================================================
# STEP 6: Select top 100 cubes (highest cube sums)
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 6: Selecting top 100 cubes")
print(f"{'='*60}")

num_spots = 100
if len(candidates) >= num_spots:
    # Last 100 after ascending sort = highest values
    selected_cubes = candidates[-num_spots:]
    print(f"Selected top {num_spots} cubes with highest cube sums")
else:
    selected_cubes = candidates
    print(f"Selected all {len(candidates)} available cubes (less than {num_spots})")

# Print statistics about selected cubes
if len(selected_cubes) > 0:
    selected_sums = [s[3] for s in selected_cubes]
    print(f"\nSelected cubes cube sum statistics:")
    print(f"  Minimum: {min(selected_sums):.2f}")
    print(f"  Maximum: {max(selected_sums):.2f}")
    print(f"  Mean: {np.mean(selected_sums):.2f}")
    print(f"  Median: {np.median(selected_sums):.2f}")

# ============================================================================
# STEP 7: Create new array with 5 channels
# Channels 0-3: Copy from groundtruth.npy
# Channel 4: Add GT spots (each spot represents entire 9x9x9 cube)
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 7: Creating groundtruth_linear.npy with 5 channels")
print(f"{'='*60}")

# Create new array with 5 channels (4 original + 1 GT channel)
# Shape: (5, n_values, n_z, y_dim, x_dim)
ground_truth_linear = np.zeros((n_channels + 1, n_values, n_z, y_dim, x_dim), dtype=data.dtype)

# Copy original 4 channels (0-3) from groundtruth.npy
ground_truth_linear[:n_channels] = data

print(f"Created new array with shape: {ground_truth_linear.shape}")
print(f"  Channels 0-3: Copied from groundtruth.npy")
print(f"  Channel 4: GT channel (will contain single voxel at center of each cube)")

# Add GT values to channel 4 (index 4)
# Each GT spot is a SINGLE voxel at the center of the 9x9x9 cube
# This single spot represents the entire cube
gt_channel_idx = n_channels  # Channel 4 (index 4)
gt_value_idx = 0  # Store GT value at value index 0 (black spot marker)

for z_center, y_center, x_center, cube_sum in selected_cubes:
    # Mark only the CENTER voxel of the cube as GT spot (black spot = 1)
    # This single spot represents the entire 9x9x9 cube
    ground_truth_linear[gt_channel_idx, gt_value_idx, z_center, y_center, x_center] = 1

# ============================================================================
# STEP 8: Save the ground truth file
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 8: Saving groundtruth_linear.npy")
print(f"{'='*60}")

np.save('groundtruth_linear.npy', ground_truth_linear)

print(f"✓ Ground truth with 5 channels saved to: groundtruth_linear.npy")
print(f"  Final shape: {ground_truth_linear.shape}")
print(f"  Channels: 0-3 (original), 4 (GT channel)")
print(f"  Number of selected cubes: {len(selected_cubes)}")
print(f"  GT spots (single voxels) in channel 4: {np.sum(ground_truth_linear[gt_channel_idx] > 0):,}")
print(f"  Each GT spot is a single voxel at the center of a {cube_size}x{cube_size}x{cube_size} cube")
print(f"  Cube distribution: Z={num_cubes_z}, Y={num_cubes_y}, X={num_cubes_x}")

print(f"\n{'='*60}")
print(f"Process completed successfully!")
print(f"{'='*60}")

Loaded data shape: (4, 11, 49, 344, 681)
  Channels: 4 (0-3)
  Values: 11 (0-10)
  Spatial dimensions: Z=49, Y=344, X=681

STEP 2: Calculating voxel intensity
Voxel intensity shape: (49, 344, 681)
  Range: [0, 8]
  Non-zero voxels: 3,845,079

STEP 3: Computing 9x9x9 cube sums (non-overlapping)
Cube size: 9x9x9
Step size: 9 (no overlap)
Number of cubes: Z=5, Y=38, X=75
Total possible cubes: 14,250
Computed 14,250 cube sums

STEP 4: GT value statistics
  Minimum GT value: 0.00
  Maximum GT value: 5034.00
  Mean GT value: 414.13
  Non-zero cubes: 6,510

STEP 5: Collecting and sorting candidates
Total candidate cubes: 6,510
  Sorted in ascending order (lowest to highest)

STEP 6: Selecting top 100 cubes
Selected top 100 cubes with highest cube sums

Selected cubes cube sum statistics:
  Minimum: 3055.00
  Maximum: 5034.00
  Mean: 3461.36
  Median: 3232.00

STEP 7: Creating groundtruth_linear.npy with 5 channels
Created new array with shape: (5, 11, 49, 344, 681)
  Channels 0-3: Copied from

visulize with groundtruth

In [28]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("groundtruth_linear.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded groundtruth_linear.npy shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3: original, 4: GT spot)")

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0 - light blue
    "#1f78b4",      # channel 1 - blue
    "#b2df8a",      # channel 2 - light green
    "#33a02c",      # channel 3 - green
    "#000000",      # channel 4 - black (GT channel)
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

# Process channels 0-3 (original channels)
for ch in range(n_channels - 1):  # Process channels 0-3
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# Process channel 4 (GT spot)
gt_channel = data[n_channels - 1]  # Channel 4
# GT spots are stored at value index 0
gt_spots = gt_channel[0]  # Shape: (n_z, y_dim, x_dim)
per_channel_3d[n_channels - 1] = gt_spots.astype(np.float32)

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.5
max_points = 5000

# Plot channels 0-3 (original channels)
for ch in range(n_channels - 1):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,
            color=channel_colors[ch],
            opacity=0.8
        ),
        name=f"Channel {ch}"
    ))

# Plot channel 4 (GT channel) - black spots
gt_channel_idx = n_channels - 1
vol = per_channel_3d[gt_channel_idx]

if np.any(vol > 0):
    z_idx, y_idx, x_idx = np.where(vol > 0)
    
    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=4,  # Larger size for GT spots
            color=channel_colors[gt_channel_idx],  # Black
            opacity=1.0,
            line=dict(width=0)
        ),
        name="GT spot"
    ))
    print(f"  GT spot: {len(z_idx)} spots")

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Visualization: All Channels (0-3) + GT spot",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
        bgcolor="white"
    ),
    autosize=True,
    showlegend=True,
)

print(f"\n3D Visualization Summary:")
print(f"  Channels 0-3: Original channels with colored markers")
print(f"  GT spot: Black spots (size=5)")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


Loaded groundtruth_linear.npy shape: (5, 11, 49, 344, 681)
  Channels: 5 (0-3: original, 4: GT spot)
  GT spot: 100 spots

3D Visualization Summary:
  Channels 0-3: Original channels with colored markers
  GT spot: Black spots (size=5)
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y
